# Quickstart: Power System Residual Calculation

This notebook demonstrates the minimal workflow for:
1. Loading power system data
2. Creating a system model
3. Evaluating power flow residuals

All with the new StructArray-based architecture - no code generation needed!

## Setup

Import required packages:

In [1]:
using Pkg
Pkg.activate("..")

using PowerFlowData
using Powerful
using Powerful.Models
using Powerful.PowerCore

  Activating project at `~/repos/Powerful`


## Step 1: Load Data

Load a test case (IEEE 14-bus system):

In [7]:
propertynames(case)

(:version, :caseid, :buses, :loads, :fixed_shunts, :generators, :branches, :transformers, :area_interchanges, :two_terminal_dc, :vsc_dc, :switched_shunts, :impedance_corrections, :multi_terminal_dc, :multi_section_lines, :zones, :area_transfers, :owners, :facts)

In [8]:
# Load raw case data
case_path = joinpath(pkgdir(Powerful), "cases", "ieee14.raw")
case = PowerFlowData.parse_network(case_path)

println("Loaded: ", case.caseid)
println("Buses: ", length(case.buses))
println("Loads: ", length(case.loads))
println("Generators: ", length(case.generators))

Loaded: CaseID(ic = 0, sbase = 100.0, rev = 33, xfrrat = 0, nxfrat = 1, basfrq = 60.0)
Buses: 14
Loads: 11
Generators: 5


## Step 2: Parse Models

Convert raw data to typed models (StructArrays):

In [10]:
# Parse individual models
bus = parse_model(Bus, case)
pq = parse_model(PQ, case)
pv = parse_model(PV, case)

println("Parsed models:")
println("  Bus: ", typeof(bus), " with ", length(bus), " instances")
println("  PQ:  ", typeof(pq), " with ", length(pq), " instances")
println("  PV:  ", typeof(pv), " with ", length(pv), " instances")

Parsed models:
  Bus: StructArrays.StructVector{BusNum{Float64}, @NamedTuple{i::Vector{Int32}, name::Vector{InlineStrings.String15}, basekv::Vector{Float64}, ide::Vector{Int8}, area::Vector{Int16}, zone::Vector{Int16}, owner::Vector{Int16}, vm::Vector{Float64}, va::Vector{Float64}, nvhi::Vector{Float64}, nvlo::Vector{Float64}, evhi::Vector{Float64}, evlo::Vector{Float64}}, Int64} with 14 instances
  PQ:  StructArrays.StructVector{PQNum{Float64}, @NamedTuple{i::Vector{Int32}, id::Vector{InlineStrings.String3}, status::Vector{Bool}, area::Vector{Int16}, zone::Vector{Int16}, pl::Vector{Float64}, ql::Vector{Float64}, ip::Vector{Float64}, iq::Vector{Float64}, yp::Vector{Float64}, yq::Vector{Float64}, owner::Vector{Int16}, scale::Vector{Union{Missing, Bool}}, intrpt::Vector{Union{Missing, Bool}}}, Int64} with 11 instances
  PV:  StructArrays.StructVector{PVNum{Float64}, @NamedTuple{i::Vector{Int32}, id::Vector{InlineStrings.String3}, pg::Vector{Float64}, qg::Vector{Float64}, qt::Vector{Float

## Step 3: Create System Model

Build the system with automatic address assignment:

In [11]:
# Create address manager
am = AddressManager()

# Create system (auto_setup=true assigns all addresses automatically)
sys = SystemModel(am, (Bus = bus, PQ = pq, PV = pv))

println("System created!")
println("  Models: ", keys(sys.models))
println("  Algebraic vars: ", am.next_idx[AlgebVar()] - 1)
println("  Algebraic residuals: ", am.next_idx[AlgebRes()] - 1)

System created!
  Models: (:Bus, :PQ, :PV)
  Algebraic vars: 28
  Algebraic residuals: 28


## Step 4: Create DAE System

Initialize the differential-algebraic equation vectors:

In [33]:
# Create DAE with initial conditions from parsed data
dae = DAE(Float64, sys)

println("DAE system:")
println("  Dimensions: m = ", dae.m)
println("  Variables (y): ", length(dae.y))
println("  Residuals (g): ", length(dae.g))
println("\nInitial conditions:")
println("  y min/max: ", minimum(dae.y), " / ", maximum(dae.y))
println("  g (should be zeros): min/max = ", minimum(dae.g), " / ", maximum(dae.g))

DAE system:
  Dimensions: m = 28
  Variables (y): 28
  Residuals (g): 28

Initial conditions:
  y min/max: -0.1669163083437297 / 1.03
  g (should be zeros): min/max = 0.0 / 0.0


## Step 5: Create Residual Workspace

Set up workspaces for efficient residual evaluation:

In [34]:
# Create system residual workspace
sys_ws = SystemResidualWorkspace(Float64, sys)


SystemResidualWorkspace{Float64, @NamedTuple{Bus::ResidualWorkspace{Float64, @NamedTuple{theta::Vector{UInt64}, v::Vector{UInt64}}, @NamedTuple{theta::Vector{Float64}, v::Vector{Float64}}, @NamedTuple{p::Vector{UInt64}, q::Vector{UInt64}}, @NamedTuple{p::Vector{Float64}, q::Vector{Float64}}}, PQ::ResidualWorkspace{Float64, @NamedTuple{theta::Vector{UInt64}, v::Vector{UInt64}}, @NamedTuple{theta::Vector{Float64}, v::Vector{Float64}}, @NamedTuple{p::Vector{UInt64}, q::Vector{UInt64}}, @NamedTuple{p::Vector{Float64}, q::Vector{Float64}}}, PV::ResidualWorkspace{Float64, @NamedTuple{theta::Vector{UInt64}, v::Vector{UInt64}}, @NamedTuple{theta::Vector{Float64}, v::Vector{Float64}}, @NamedTuple{p::Vector{UInt64}, v_ctrl::Vector{UInt64}}, @NamedTuple{p::Vector{Float64}, v_ctrl::Vector{Float64}}}}, @NamedTuple{Bus::StructArrays.StructVector{BusNum{Float64}, @NamedTuple{i::Vector{Int32}, name::Vector{InlineStrings.String15}, basekv::Vector{Float64}, ide::Vector{Int8}, area::Vector{Int16}, zone::

In [35]:
propertynames(sys_ws.model_workspaces)

println("Residual workspace created")

Residual workspace created


In [38]:
dae.g

28-element Vector{Float64}:
  81.442
  18.3
 -10.0
 -47.8
  -7.6
  15.0
   0.0
  35.0
 -29.5
  -9.0
   ⋮
  -7.53129
   0.0
  -0.011050000000000004
 -16.6
  -5.8
  -1.8
  -1.6
  -5.8
  -7.0

## Step 6: Evaluate Residuals

Calculate the power flow residuals:

In [37]:
# Evaluate residuals at current DAE state
evaluate_system_residuals!(sys_ws, dae)

println("Residuals evaluated!")
println("  Max residual: ", maximum(abs.(dae.g)))
println("  RMS residual: ", sqrt(sum(dae.g.^2) / length(dae.g)))
println("\nResidual statistics:")
println("  Any non-zero? ", any(dae.g .!= 0.0))
println("  Number of non-zero: ", count(dae.g .!= 0.0))

Residuals evaluated!
  Max residual: 81.442
  RMS residual: 22.190088386672127

Residual statistics:
  Any non-zero? true
  Number of non-zero: 25


## Step 7: Inspect Residuals

Look at individual model contributions:

In [30]:
# Get PQ load workspace
pq_ws = sys_ws.model_workspaces[:PQ]

println("PQ Load residuals:")
println("  P residuals: ", pq_ws.res_vals.p)
println("  Q residuals: ", pq_ws.res_vals.q)
println("\nPQ loads consume power (negative injection):")
println("  All P < 0? ", all(pq_ws.res_vals.p .< 0))
println("  All Q < 0? ", all(pq_ws.res_vals.q .< 0))

PQ Load residuals:
  P residuals: [-21.7, -50.0, -47.8, -7.6, -15.0, -29.5, -9.0, -3.5, -6.1, -13.5, -20.0]
  Q residuals: [-12.7, -25.0, -10.0, -1.6, -7.5, -16.6, -5.8, -1.8, -1.6, -5.8, -7.0]

PQ loads consume power (negative injection):
  All P < 0? true
  All Q < 0? true


In [31]:
# Get PV generator workspace
pv_ws = sys_ws.model_workspaces[:PV]

println("PV Generator residuals:")
println("  P residuals: ", pv_ws.res_vals.p)
println("\nPV generators inject power (positive):")
println("  All P > 0? ", all(pv_ws.res_vals.p .> 0))

PV Generator residuals:
  P residuals: [81.442, 40.0, 40.0, 30.0, 35.0]

PV generators inject power (positive):
  All P > 0? true


## Step 8: Visualize Variable Mapping

See how variables map to DAE vectors:

In [32]:
# Bus theta addresses
theta_addrs = am.assignments[:Bus].by_type[AlgebVar()][:theta]
println("Bus theta addresses: ", theta_addrs)
println("Theta values from DAE: ", dae.y[theta_addrs])

# Bus voltage addresses  
v_addrs = am.assignments[:Bus].by_type[AlgebVar()][:v]
println("\nBus v addresses: ", v_addrs)
println("Voltage values from DAE: ", dae.y[v_addrs])

Bus theta addresses: UInt64[0x0000000000000001, 0x0000000000000002, 0x0000000000000003, 0x0000000000000004, 0x0000000000000005, 0x0000000000000006, 0x0000000000000007, 0x0000000000000008, 0x0000000000000009, 0x000000000000000a, 0x000000000000000b, 0x000000000000000c, 0x000000000000000d, 0x000000000000000e]
Theta values from DAE: [0.0, -0.02798111856797309, -0.06009692213392075, -0.07472103593638124, -0.06431538293599104, -0.10999763077769062, -0.08428544023731016, -0.024338616419060925, -0.12750153784594176, -0.130201562198777, -0.12294797382748855, -0.1289344531618291, -0.13378646848237333, -0.1669163083437297]

Bus v addresses: UInt64[0x000000000000000f, 0x0000000000000010, 0x0000000000000011, 0x0000000000000012, 0x0000000000000013, 0x0000000000000014, 0x0000000000000015, 0x0000000000000016, 0x0000000000000017, 0x0000000000000018, 0x0000000000000019, 0x000000000000001a, 0x000000000000001b, 0x000000000000001c]
Voltage values from DAE: [1.03, 1.0197, 1.00042, 0.99858, 1.00443, 0.99871,

## Summary

Complete workflow in 8 simple steps:

1. ✅ Load data with PowerFlowData.jl
2. ✅ Parse to StructArray models (no code generation!)
3. ✅ Create SystemModel (auto address assignment)
4. ✅ Initialize DAE vectors
5. ✅ Build residual workspace
6. ✅ Evaluate residuals (zero-allocation)
7. ✅ Inspect results per model
8. ✅ Map between models and DAE vectors

### Key Benefits

- **Simple API**: Just `SystemModel(am, models)` - everything is automatic
- **Type stable**: StructArrays available at compile time
- **Fast**: Struct-of-arrays layout, zero-allocation residual eval
- **Clean**: No generated types, no boilerplate